In [4]:
from ultralytics import YOLO
import cv2
import numpy as np
import pandas as pd
import os

model = YOLO("best.pt")

IMAGE_FOLDER = "akshay_separate.v1i.yolov8/train/images"
data = []

In [18]:
data = []

for img in os.listdir(IMAGE_FOLDER):

    path = os.path.join(IMAGE_FOLDER, img)

    results = model(path)
    r = results[0]

    if r.masks is None:
        continue

    image = cv2.imread(path)
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)

    for mask_tensor in r.masks.data:

        mask = mask_tensor.cpu().numpy()

        mask = cv2.resize(mask, (gray.shape[1], gray.shape[0]))
        mask = mask > 0.5

        pixels = gray[mask]

        if len(pixels) == 0:
            continue

        mean_T = np.mean(pixels)
        std_T = np.std(pixels)

        threshold = mean_T + 1.5 * std_T
        hotspot_ratio = np.sum(pixels > threshold) / len(pixels)

        gradient = cv2.Laplacian(gray, cv2.CV_64F)
        gradient_values = np.abs(gradient[mask])

        gradient_mean = np.mean(gradient_values)

        data.append([
            img,
            mean_T,
            std_T,
            hotspot_ratio,
            gradient_mean
        ])


image 1/1 c:\Users\aksha\OneDrive\Desktop\Mango_project\akshay_separate.v1i.yolov8\train\images\FLIR0013_jpg.rf.0119080546aa760cfdfc84d2738c0370.jpg: 640x640 1 akshay_separate - v1 2026-02-27 12-40pm, 215.4ms
Speed: 3.4ms preprocess, 215.4ms inference, 2.4ms postprocess per image at shape (1, 3, 640, 640)

image 1/1 c:\Users\aksha\OneDrive\Desktop\Mango_project\akshay_separate.v1i.yolov8\train\images\FLIR0013_jpg.rf.0b1598ae9baf8a94b6d21204603c6277.jpg: 640x640 1 akshay_separate - v1 2026-02-27 12-40pm, 195.8ms
Speed: 3.8ms preprocess, 195.8ms inference, 2.1ms postprocess per image at shape (1, 3, 640, 640)

image 1/1 c:\Users\aksha\OneDrive\Desktop\Mango_project\akshay_separate.v1i.yolov8\train\images\FLIR0013_jpg.rf.0fb2117eb0c212e3b4b7d441acb2dad3.jpg: 640x640 1 akshay_separate - v1 2026-02-27 12-40pm, 195.7ms
Speed: 3.5ms preprocess, 195.7ms inference, 2.1ms postprocess per image at shape (1, 3, 640, 640)

image 1/1 c:\Users\aksha\OneDrive\Desktop\Mango_project\akshay_separate.v1i

In [21]:
df = pd.DataFrame(data, columns=[
    "image",
    "mean_T",
    "std_T",
    "hotspot_ratio",
    "gradient_mean"
])

df.to_csv("defect_feature.csv", index=False)

In [13]:
df["defect_score"] = (
    0.5 * df["hotspot_ratio"] +
    0.3 * df["gradient_mean"] +
    0.2 * df["std_T"]
)

threshold = df["defect_score"].median()

df["defect"] = df["defect_score"].apply(
    lambda x: 1 if x > threshold else 0
)

In [14]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier

X = df[["mean_T","std_T","hotspot_ratio","gradient_mean"]]
y = df["defect"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y
)

model = RandomForestClassifier(n_estimators=200)
model.fit(X_train, y_train)

ValueError: With n_samples=1, test_size=0.2 and train_size=None, the resulting train set will be empty. Adjust any of the aforementioned parameters.

In [19]:
print(len(df))
# print(df)

1


In [17]:
df.head()

,image,mean_T,std_T,hotspot_ratio,gradient_mean,defect_score,defect
0,FLIR1399_jpg.rf.f621b58711fda8496ea49746310ddf...,218.546126,30.663554,0.0,3.237734,7.104031,0
